# 02 — Robot Data Quality
Quality flags and safe split-gap repair.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if not (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DataConfig
from src.data_quality import run_quality_pipeline


In [2]:
raw_stocks = pd.read_parquet(
    PROJECT_ROOT / "data" / "raw" / "bist100_robot_raw.parquet"
)
raw_market = pd.read_parquet(
    PROJECT_ROOT / "data" / "raw" / "xu100_robot_raw.parquet"
)

config = DataConfig()
stock_quality = run_quality_pipeline(raw_stocks, config, apply_split_repairs=True)
market_quality = run_quality_pipeline(raw_market, config, apply_split_repairs=True)

display(stock_quality.summary.head(20))
display(stock_quality.split_repairs)


,Ticker,StartDate,EndDate,RowCount,ValidPriceBars,TradableBars,MissingRows,NonPositivePriceRows,InvalidOHLCRows,ZeroVolumeRows,SyntheticNoTradeBars,DividendEvents,SplitEvents,SuspectedSplitGaps,LargeCloseReturns,ValidBarRatio,TradableBarRatio
0,MIATK.IS,2021-11-22,2026-07-24,1177,1171,1167,5,0,1,9,4,0,1,0,0,0.994902,0.991504
1,PSGYO.IS,2021-12-16,2026-07-24,1159,1154,1149,4,0,1,9,5,4,3,0,0,0.995686,0.991372
2,SASA.IS,2018-01-01,2026-07-24,2169,2165,2025,4,0,0,144,26,1,6,0,0,0.998156,0.933610
3,ASELS.IS,2018-01-01,2026-07-24,2169,2164,2128,5,0,0,41,36,14,2,0,0,0.997695,0.981097
4,OYAKC.IS,2018-01-01,2026-07-24,2169,2165,2134,4,0,0,35,31,3,1,0,0,0.998156,0.983864
5,OTKAR.IS,2018-01-01,2026-07-24,2169,2162,2137,7,0,0,32,25,6,1,0,0,0.996773,0.985247
6,TUKAS.IS,2018-01-01,2026-07-24,2169,2165,2137,4,0,0,32,28,0,3,0,0,0.998156,0.985247
7,RALYH.IS,2018-01-01,2026-07-24,2169,2165,2138,4,0,0,31,27,0,2,0,0,0.998156,0.985708
8,TCELL.IS,2018-01-01,2026-07-24,2169,2164,2138,5,0,0,31,25,14,0,0,0,0.997695,0.985708
9,TRENJ.IS,2018-01-01,2026-07-24,2169,2164,2138,5,0,0,31,26,0,0,0,0,0.997695,0.985708


,Ticker,Date,RawFactor,EstimatedSplitRatio,AppliedFactor,RelativeError,YahooSplitValue
0,BSOKE.IS,2024-12-02,0.261728,4,0.250000,0.046914,0.0
1,CCOLA.IS,2024-08-01,0.091769,11,0.090909,0.009456,0.0
2,FENER.IS,2025-06-23,0.215061,5,0.200000,0.075307,0.0
3,GSRAY.IS,2024-12-23,0.329125,3,0.333333,0.012626,0.0
4,KTLEV.IS,2025-01-02,0.033833,30,0.033333,0.014979,0.0
5,TUKAS.IS,2025-02-03,0.329271,3,0.333333,0.012188,0.0


In [3]:
processed_dir = PROJECT_ROOT / "data" / "processed"
results_dir = PROJECT_ROOT / "results"
processed_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

stock_quality.enriched.to_parquet(
    processed_dir / "bist100_robot_enriched.parquet", index=False
)
stock_quality.clean.to_parquet(
    processed_dir / "bist100_robot_clean.parquet", index=False
)
market_quality.clean.to_parquet(
    processed_dir / "xu100_robot_clean.parquet", index=False
)
stock_quality.summary.to_csv(
    results_dir / "data_quality_summary.csv", index=False
)
stock_quality.split_repairs.to_csv(
    results_dir / "split_repair_log.csv", index=False
)
